# **EDA Notebook**

---
## 0. Setup Environment

In [1]:
# DO NOT MODIFY THE CODE IN THIS CELL
!pip install -q utstd
from utstd.folders import *
from utstd.ipyrenders import *
at = AtFolder(course_code=36106, assignment="AT3")
at.run()
import warnings
warnings.simplefilter(action='ignore')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.9/12.9 MB 95.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 100.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
umap-learn 0.5.12 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
hdbscan 0.8.42 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.
Mounted at /content/gdrive

You can now save your data files in: /content/gdrive/MyDrive/36106/assignment/AT3/data


---
## Student Information

In [2]:
group_name = "Neural Minds"
student_name = "Pal Patel"
student_id = "26173702"

In [3]:
# Do not modify
print_tile(size='h1', key='group_name', value=group_name)

In [4]:
# Do not modify
print_tile(size='h1', key='student_name', value=student_name)

In [5]:
# Do not modify
print_tile(size='h1', key='student_id', value=student_id)

---
## 0. Python Packages

### 0.a Install Additional Packages

In [ ]:
# No additional packages required

### 0.b Import Packages

In [6]:
import pandas as pd
import altair as alt
import numpy as np

---
## B. Data Understanding

In [7]:
# Do not modify
try:
  df = pd.read_csv(at.folder_path / 'product_list_price_history.csv')
except Exception as e:
  print(e)

### B.1 Explore Dataset

In [9]:
print('Shape:', df.shape)
print('Columns:', list(df.columns))
df.head(5)

Shape: (620, 4)
Columns: ['product_id', 'start_date', 'end_date', 'list_price']


,product_id,start_date,end_date,list_price
0,d8f1ff26-6a35-4e95-814b-de35bdd87ce0,2012-05-30 00:00:00,2013-05-29 00:00:00,44.9900
1,ce0a0304-eeaf-4032-b4ae-1a31c74ce76b,2013-05-30 00:00:00,NaN,539.9900
2,9014b398-fe53-4fb0-8177-97347cea83c3,2013-05-30 00:00:00,NaN,2294.9900
3,05385566-38df-4855-94a8-786cae6ffe49,2011-05-31 00:00:00,2012-05-29 00:00:00,33.6442
4,9e4e275e-baa1-44e2-827e-ec3fead348fa,2013-05-30 00:00:00,NaN,NaN


In [10]:
print(df.dtypes)
print('Missing:', df.isnull().sum())
print('Unique products:', df['product_id'].nunique())
df.describe()

product_id     object
start_date     object
end_date       object
list_price    float64
dtype: object
Missing: product_id      0
start_date     22
end_date      311
list_price    145
dtype: int64
Unique products: 293


,list_price
count,475.000000
mean,736.673149
std,848.743137
min,2.290000
25%,60.745000
50%,348.760000
75%,1120.490000
max,3578.270000


In [11]:
dataset_insights = """
product_list_price_history has 620 rows and 4 columns tracking list price changes per product.
Columns: product_id, start_date, end_date, list_price.
Missing: start_date 22 (3.5%), end_date 311 (50%), list_price 145 (23%).
The 145 missing list_price values are critical - those rows cannot be used in modelling.
list_price ranges from $2.29 to $3578.27 with mean $736.67 - strongly right-skewed.
Primary dataset for clustering as list price defines product market positioning.
"""

In [12]:
# Do not modify
print_tile(size='h3', key='dataset_insights', value=dataset_insights)

### B.2 Explore Feature of Interest `list_price`

In [13]:
print(df['list_price'].describe())
alt.Chart(df.dropna(subset=['list_price'])).mark_bar(color='royalblue').encode(
    x=alt.X('list_price:Q', bin=alt.Bin(maxbins=30), title='List Price ($)'),
    y='count():Q'
).properties(title='Distribution of List Price', width=450, height=300)

count     475.000000
mean      736.673149
std       848.743137
min         2.290000
25%        60.745000
50%       348.760000
75%      1120.490000
max      3578.270000
Name: list_price, dtype: float64


alt.Chart(...)

In [14]:
feature_1_insights = """
list_price is the retail selling price per product per time period.
Distribution is right-skewed with two clusters: low-price accessories and high-price bikes.
145 rows with missing list_price will be dropped as price is essential for clustering.
Aggregated avg_price per product is the single most important clustering feature.
"""

In [15]:
# Do not modify
print_tile(size='h3', key='feature_1_insights', value=feature_1_insights)

### B.3 Explore Feature of Interest `Price Volatility` (derived)

In [16]:
pv = df.groupby('product_id')['list_price'].agg(['std','max','min','count']).reset_index()
pv.columns = ['product_id','price_std','max_price','min_price','n_records']
pv['price_change'] = pv['max_price'] - pv['min_price']
pv['price_std'] = pv['price_std'].fillna(0)
print(pv['price_std'].describe())
alt.Chart(pv).mark_bar(color='mediumseagreen').encode(
    x=alt.X('price_std:Q', bin=alt.Bin(maxbins=25), title='Price Std Dev ($)'),
    y='count():Q'
).properties(title='Price Volatility per Product', width=450, height=280)

count    293.000000
mean      11.505841
std       32.589826
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max      185.111716
Name: price_std, dtype: float64


alt.Chart(...)

In [17]:
feature_2_insights = """
Price volatility (std dev per product) captures list price instability over time.
High volatility indicates multiple price changes typical of competitive premium items.
Low volatility means stable pricing typical of commodity or entry-level accessories.
price_std and price_change are key clustering features differentiating dynamic vs stable products.
"""

In [18]:
# Do not modify
print_tile(size='h3', key='feature_2_insights', value=feature_2_insights)

### B.4 Explore Feature of Interest `Margin` (derived)

In [19]:
try:
  cdf = pd.read_csv(at.folder_path / 'product_cost_history.csv')
except Exception as e:
  print(e)
price_agg = df.groupby('product_id')['list_price'].mean().reset_index()
price_agg.columns = ['product_id','avg_price']
cost_agg = cdf.groupby('product_id')['standard_cost'].mean().reset_index()
cost_agg.columns = ['product_id','avg_cost']
m = price_agg.merge(cost_agg, on='product_id').dropna()
m['margin_pct'] = ((m['avg_price'] - m['avg_cost']) / m['avg_price']) * 100
print(m['margin_pct'].describe())
alt.Chart(m).mark_point(opacity=0.5, color='purple').encode(
    x=alt.X('avg_cost:Q', title='Avg Cost ($)'),
    y=alt.Y('avg_price:Q', title='Avg Price ($)'),
    tooltip=['avg_cost','avg_price','margin_pct']
).properties(title='Cost vs Price per Product', width=420, height=300)

count    253.000000
mean      47.129769
std        9.411928
min       23.000000
25%       40.059995
50%       45.239996
75%       55.599983
max       64.249474
Name: margin_pct, dtype: float64


alt.Chart(...)

In [20]:
feature_n_insights = """
Margin percentage is the most business-critical derived feature for clustering.
Strong linear relationship between cost and price but margin varies significantly within tiers.
Products with margins above 50% are likely premium bikes with strong brand positioning.
Low-margin products may be accessories or loss-leaders used to drive volume.
margin_pct and price_to_cost_ratio directly translate cluster results into business strategy.
"""

In [21]:
# Do not modify
print_tile(size='h3', key='feature_n_insights', value=feature_n_insights)